# SARAI - Modele 2 : Semantic Search Chatbot

**Stocktaking of Arab Regional AI Initiatives**

Chatbot base sur la **recherche semantique** (sentence-transformers + cosine similarity)
qui comprend le sens des questions et retrouve les informations les plus pertinentes.

**AUCUN LLM utilise** : Pas de GPT, Claude, Mistral, Llama, Gemini.
Seulement `sentence-transformers/all-MiniLM-L6-v2` (384 dimensions).

---
## Architecture

```
Question → Query Parser (stop words, intent, pays, secteur, tech)
         → Embedding de la question (all-MiniLM-L6-v2)
         → Cosine Similarity vs toutes les entites embarquees
         → Scoring (similarite cosinus)
         → Build Response (FR/EN)
```

Compare a Modele 1 (Keyword Search) :
- Modele 1 : matche des mots exacts (ILIKE) → ne trouve pas si mots differents
- Modele 2 : comprend le sens → trouve des resultats meme avec des mots differents
- Exemple : "AI projects in UAE" → Modele 1 echoue (pas de 'ai' dans les titres UAE), Modele 2 trouve (comprend le sens)

In [1]:
# ── Imports ──
import sys, os, re, json, time, warnings
from urllib.parse import quote_plus
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker
import numpy as np
warnings.filterwarnings('ignore')

print('Imports OK')

Imports OK


In [2]:
# ── Connexion DB (PostgreSQL -> fallback SQLite) ──

cwd = os.getcwd()
candidates = [
    os.path.join(cwd, '..', 'backend'),
    os.path.join(cwd, 'backend'),
    os.path.abspath('../backend'),
    os.path.abspath('backend'),
]

backend_dir = None
for d in candidates:
    d = os.path.abspath(d)
    if os.path.exists(os.path.join(d, 'sarai.db')):
        backend_dir = d
        break

if backend_dir is None:
    raise FileNotFoundError(
        "Base de donnees introuvable. Cherche dans: "
        + ", ".join(os.path.abspath(d) for d in candidates)
    )

print(f'[DB] Backend trouve: {backend_dir}')

env_path = os.path.join(backend_dir, '.env')
if os.path.exists(env_path):
    load_dotenv(env_path)

password = os.getenv('DB_PASSWORD', '0000')
db_user = os.getenv('DB_USER', 'postgres')
db_host = os.getenv('DB_HOST', 'localhost')
db_port = os.getenv('DB_PORT', '5432')
db_name = os.getenv('DB_NAME', 'SARAI_DB')

SQLITE_PATH = os.path.join(backend_dir, 'sarai.db')
SQLITE_URL = f'sqlite:///{SQLITE_PATH}'

engine = None
if db_host:
    DATABASE_URL = f'postgresql+psycopg://{db_user}:{quote_plus(password)}@{db_host}:{db_port}/{db_name}'
    try:
        test_engine = create_engine(DATABASE_URL, pool_pre_ping=True)
        with test_engine.connect() as conn:
            conn.execute(text('SELECT 1'))
        engine = create_engine(DATABASE_URL, pool_pre_ping=True)
        print(f'[DB] PostgreSQL OK: {db_host}:{db_port}/{db_name}')
    except Exception as e:
        print(f'[DB] PostgreSQL indisponible: {e}')

if engine is None:
    print(f'[DB] SQLite: {SQLITE_PATH}')
    engine = create_engine(SQLITE_URL, connect_args={'check_same_thread': False})

SessionLocal = sessionmaker(bind=engine, autoflush=False, autocommit=False)
db = SessionLocal()

# Verifier les donnees
for t in ['projects', 'stakeholders', 'resources']:
    try:
        c = db.execute(text(f'SELECT COUNT(*) FROM {t}')).scalar()
        print(f'[DB] {t}: {c} lignes')
    except Exception as ex:
        print(f'[DB] {t}: ABSENTE - {ex}')

# Apercu
try:
    row = db.execute(text('SELECT p.id, p.title, p.sector, c.country FROM projects p LEFT JOIN countries c ON c.id=p.country_id LIMIT 5')).fetchall()
    print('\nApercu projets:')
    for r in row:
        print(f'  {r.id}: {r.title} | {r.sector} | {r.country}')
except Exception as ex:
    print(f'Apercu non disponible: {ex}')

[DB] Backend trouve: c:\Users\user\OneDrive - ESPRIT\Documents\GitHub\test\Stage-PFE-AICTO\backend
[DB] PostgreSQL indisponible: No module named 'psycopg'
[DB] SQLite: c:\Users\user\OneDrive - ESPRIT\Documents\GitHub\test\Stage-PFE-AICTO\backend\sarai.db
[DB] projects: 12 lignes
[DB] stakeholders: 12 lignes
[DB] resources: 5 lignes

Apercu projets:
  1: AI Diagnostic System | Health | Egypt
  2: Smart Irrigation System | Agriculture | Morocco
  3: Adaptive Learning Platform | Education | UAE
  4: Traffic Management AI | Transportation | Saudi Arabia
  5: Financial Fraud Detection | Finance | UAE


---
## 1. Dictionnaires et Query Parser

(Identique au Modele 1 pour la compatibilite de l'evaluation)

In [3]:
# ── Dictionnaires ──

INTENT_WORDS = {
    'project','projects','projet','projets','initiative','initiatives',
    'stakeholder','stakeholders','organization','organizations','organisation','organisations',
    'resource','resources','ressource','ressources',
    'startup','startups','ngo','ngos','company','companies','lab','labs','center','centre',
    'report','reports','dataset','datasets','publication','publications','tool','tools','guide','guides',
    'show','shows','list','liste','give','gives','find','finds','search',
    'montre','montrer','moi','donne','donner','lister','affiche','afficher',
    'cherche','chercher','trouve','trouver','veux','peux','peut',
    'all','tous','toutes','les','des',
    'quels','quelles','quel','quelle','me','you','your','my',
}

STOP_WORDS = {
    'le','la','les','des','de','du','un','une','et','est','sont',
    'dans','pour','sur','avec','par','pas','que','qui','quoi',
    'the','a','an','in','on','at','to','for','of','and','or',
    'is','are','was','were','be','been','being','have','has',
    'had','do','does','did','will','would','could','should',
    'may','might','shall','can','need','dare','ought','used',
    'what','which','who','whom','this','that','these','those',
    'am','it','its','some','any','each','every','all','both',
    'few','more','most','other','such','no','nor','not','only',
    'own','same','so','than','too','very','just','because','as',
    'until','while','if','else','when','where','why','how',
    'about','between','through','during','before','after','above',
    'below','up','down','out','off','over','under','again',
    'further','then','once','here','there','en','y','a','au','aux',
    'avoir','etre','faire',
}

COUNTRY_MAP = {
    'tunisia':'Tunisia','tunisie':'Tunisia','algeria':'Algeria','algerie':'Algeria',
    'morocco':'Morocco','maroc':'Morocco','egypt':'Egypt','egypte':'Egypt',
    'uae':'UAE','emirates':'UAE','dubai':'UAE','saudi':'Saudi Arabia',
    'qatar':'Qatar','kuwait':'Kuwait','oman':'Oman','bahrain':'Bahrain',
    'lebanon':'Lebanon','liban':'Lebanon','jordan':'Jordan','iraq':'Iraq',
    'yemen':'Yemen','syria':'Syria','palestine':'Palestine',
    'mauritania':'Mauritania','libya':'Libya','sudan':'Sudan','somalia':'Somalia',
    'djibouti':'Djibouti','comoros':'Comoros',
}

ENTITY_MAP = {
    'project': {'project','projects','projet','projets','initiative','initiatives'},
    'stakeholder': {'stakeholder','stakeholders','organization','organizations',
                    'organisation','organisations','startup','startups',
                    'ngo','ngos','company','companies','lab','labs'},
    'resource': {'resource','resources','ressource','ressources',
                 'report','reports','dataset','datasets'},
}

SECTOR_MAP = {
    'health': {'health','healthcare','medical','sante','hospital','telemedicine'},
    'education': {'education','school','university','training','learning','edutech','adaptive'},
    'agriculture': {'agriculture','agricultural','farming','food','agritech','irrigation'},
    'finance': {'finance','financial','banking','fintech','fraud'},
    'energy': {'energy','renewable','solar','wind','power'},
    'environment': {'environment','environmental','climate','water','green','waste'},
    'transportation': {'transportation','transport','traffic','logistics'},
    'security': {'security','cyber','cybersecurity'},
}

TECH_MAP = {
    'nlp':'NLP','natural language':'NLP','llm':'NLP',
    'computer vision':'Computer Vision','vision':'Computer Vision',
    'machine learning':'Machine Learning','ml':'Machine Learning',
    'deep learning':'Deep Learning','dl':'Deep Learning',
    'robotics':'Robotics','robot':'Robotics','blockchain':'Blockchain',
    'iot':'IoT','internet of things':'IoT','big data':'Big Data',
    'generative':'Generative AI','genai':'Generative AI',
    'speech recognition':'Speech Recognition','speech':'Speech Recognition',
}

print(f'Dictionnaires charges: {len(COUNTRY_MAP)} pays, {len(ENTITY_MAP)} entites, {len(SECTOR_MAP)} secteurs, {len(TECH_MAP)} technologies')

Dictionnaires charges: 29 pays, 3 entites, 8 secteurs, 19 technologies


In [4]:
# ── Fonctions de parsing (identiques au Modele 1) ──

def extract_keywords(query):
    lower = query.lower().strip().replace('-',' ').replace("'",' ').replace('_',' ')
    tokens = re.split(r"[\s,;:!?()]+", lower)
    return [t for t in tokens if len(t) > 1
            and t not in STOP_WORDS
            and t not in INTENT_WORDS]

def detect_country(query):
    lower = query.lower()
    for alias, country in COUNTRY_MAP.items():
        if alias in lower:
            return country
    return None

def detect_entity(query):
    lower = query.lower()
    for entity, keywords in ENTITY_MAP.items():
        for kw in keywords:
            if kw in lower:
                return entity
    return None

def detect_sector(query):
    lower = query.lower()
    for sector, keywords in SECTOR_MAP.items():
        for kw in keywords:
            if kw in lower:
                return sector.capitalize()
    return None

def detect_tech(query):
    lower = query.lower()
    for alias, tech in TECH_MAP.items():
        if alias in lower:
            return tech
    return None

def detect_lang(query):
    french = {'quels','quelles','quel','quelle','projets','sante','tunisie',
              'maroc','donne','montre','sante'}
    tokens = set(re.split(r"[\s,;:!?()]+", query.lower().strip()))
    arabic_chars = set('ابتثجحخدذرزسشصضطظعغفقكلمنهويآأؤإئ')
    if any(c in query for c in arabic_chars):
        return 'ar'
    if tokens & french:
        return 'fr'
    return 'en'

def parse_query(query):
    return {
        'original': query,
        'keywords': extract_keywords(query),
        'country': detect_country(query),
        'entity': detect_entity(query),
        'sector': detect_sector(query),
        'technology': detect_tech(query),
        'language': detect_lang(query),
    }

# Test du parser
test_queries = [
    'Quels projets IA en sante en Tunisie ?',
    'NLP projects in Morocco',
    'Show me AI startups in Egypt',
    'Machine learning resources for healthcare',
    'AI healthcare projects',
    'Arabic speech recognition',
]
print('Test du Query Parser:')
print('='*70)
for q in test_queries:
    p = parse_query(q)
    print(f'\n{q}')
    print(f'  keywords={p["keywords"]}')
    print(f'  pays={p["country"]} | type={p["entity"]} | secteur={p["sector"]} | tech={p["technology"]} | lang={p["language"]}')

Test du Query Parser:

Quels projets IA en sante en Tunisie ?
  keywords=['ia', 'sante', 'tunisie']
  pays=Tunisia | type=project | secteur=Health | tech=None | lang=fr

NLP projects in Morocco
  keywords=['nlp', 'morocco']
  pays=Morocco | type=project | secteur=None | tech=NLP | lang=en

Show me AI startups in Egypt
  keywords=['ai', 'egypt']
  pays=Egypt | type=stakeholder | secteur=None | tech=None | lang=en

Machine learning resources for healthcare
  keywords=['machine', 'learning', 'healthcare']
  pays=None | type=resource | secteur=Health | tech=Machine Learning | lang=en

AI healthcare projects
  keywords=['ai', 'healthcare']
  pays=None | type=project | secteur=Health | tech=None | lang=en

Arabic speech recognition
  keywords=['arabic', 'speech', 'recognition']
  pays=None | type=None | secteur=None | tech=Speech Recognition | lang=en


---
## 2. Generation des Embeddings

Utilise `sentence-transformers/all-MiniLM-L6-v2` (384 dimensions) pour
transformer chaque entite (projet, stakeholder, ressource) en vecteur.

Chaque contenu est une concatenation de :
- Projets : titre | description | secteur | technologie | organisation
- Stakeholders : nom | description | type | pays
- Ressources : titre | description | type | categorie

In [5]:
# ── Chargement des donnees depuis la DB ──

def load_entities(session):
    """Charge toutes les entites (projets, stakeholders, ressources) depuis la DB."""
    entities = []

    # Projets
    rows = session.execute(text('''
        SELECT p.id, p.title, p.description, p.sector, p.technology,
               p.organization, p.year_of_implementation,
               c.country AS country
        FROM projects p
        LEFT JOIN countries c ON c.id = p.country_id
        WHERE p.status NOT IN ('pending','rejected')
    ''')).fetchall()

    for r in rows:
        content = ' | '.join(filter(None, [
            r.title, r.description, r.sector, r.technology, r.organization, r.country
        ]))
        entities.append({
            'type': 'project',
            'id': r.id,
            'title': r.title,
            'country': r.country or '',
            'sector': r.sector or '',
            'technology': r.technology or '',
            'organization': r.organization or '',
            'description': (r.description or '')[:300],
            'content': content,
        })

    # Stakeholders
    rows = session.execute(text('''
        SELECT id, name, description, type, country, category
        FROM stakeholders
    ''')).fetchall()

    for r in rows:
        content = ' | '.join(filter(None, [
            r.name, r.description, r.type, r.country, r.category
        ]))
        entities.append({
            'type': 'stakeholder',
            'id': r.id,
            'title': r.name,
            'country': r.country or '',
            'sector': r.type or '',
            'technology': '',
            'organization': '',
            'description': (r.description or '')[:300],
            'content': content,
        })

    # Ressources
    rows = session.execute(text('''
        SELECT id, title, description, type, category, language
        FROM resources
    ''')).fetchall()

    for r in rows:
        content = ' | '.join(filter(None, [
            r.title, r.description, r.type, r.category, r.language
        ]))
        entities.append({
            'type': 'resource',
            'id': r.id,
            'title': r.title,
            'country': '',
            'sector': r.category or '',
            'technology': '',
            'organization': '',
            'description': (r.description or '')[:300],
            'content': content,
        })

    print(f'[DATA] {len(entities)} entites chargees')
    print(f'  - {sum(1 for e in entities if e["type"]=="project")} projets')
    print(f'  - {sum(1 for e in entities if e["type"]=="stakeholder")} stakeholders')
    print(f'  - {sum(1 for e in entities if e["type"]=="resource")} ressources')

    return entities


entities = load_entities(db)

[DATA] 26 entites chargees
  - 9 projets
  - 12 stakeholders
  - 5 ressources


In [6]:
# ── Generation des embeddings avec sentence-transformers ──

print('[EMB] Chargement du modele sentence-transformers/all-MiniLM-L6-v2...')
start = time.time()

from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

print(f'[EMB] Modele charge en {time.time()-start:.1f}s')
print(f'[EMB] Dimension des embeddings: {embedding_model.get_sentence_embedding_dimension()}')

# Embedding de toutes les entites
texts = [e['content'] for e in entities]
print(f'[EMB] Generation de {len(texts)} embeddings...')
start = time.time()

entity_embeddings = embedding_model.encode(texts, normalize_embeddings=True, show_progress_bar=True)

print(f'[EMB] Embeddings generes en {time.time()-start:.1f}s')
print(f'[EMB] Forme: {entity_embeddings.shape}')

# Stocker les embeddings dans chaque entite
for i, e in enumerate(entities):
    e['embedding'] = entity_embeddings[i]

print('[EMB] Embeddings prets pour la recherche semantique')

[EMB] Chargement du modele sentence-transformers/all-MiniLM-L6-v2...

[EMB] Modele charge en 24.3s
[EMB] Dimension des embeddings: 384
[EMB] Generation de 26 embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[EMB] Embeddings generes en 0.4s
[EMB] Forme: (26, 384)
[EMB] Embeddings prets pour la recherche semantique


---
## 3. Moteur de Recherche Semantique

Convertit la question en embedding et calcule la **similarite cosinus**
avec toutes les entites. Retourne les plus similaires.

Avantage vs Modele 1 :
- Comprend le sens ("NLP projects in Morocco" → trouve "Adaptive Learning Platform")
- Pas besoin de mots exacts
- Robustesse aux fautes d'orthographe et synonymes

In [7]:
# ── Moteur de recherche semantique ──

def semantic_search(
    query_entities,      # liste d'entites avec 'embedding'
    query_embedding,     # embedding de la question (numpy array)
    entity_type=None,    # filtrer par type: 'project','stakeholder','resource' ou None
    country=None,        # filtrer par pays
    sector=None,         # filtrer par secteur
    tech=None,           # filtrer par technologie
    limit=10
):
    """Recherche semantique : cosine similarity + filtres."""
    scored = []

    for e in query_entities:
        if entity_type and e['type'] != entity_type:
            continue
        if country and country.lower() not in e.get('country', '').lower():
            continue
        if sector and sector.lower() not in e.get('sector', '').lower():
            continue
        if tech and tech.lower() not in e.get('technology', '').lower():
            continue

        # Cosine similarity (vecteurs deja normalises -> produit scalaire)
        sim = float(np.dot(query_embedding, e['embedding']))
        scored.append((sim, e))

    scored.sort(key=lambda x: x[0], reverse=True)
    total = len(scored)
    return [e for _, e in scored[:limit]], total

print('Moteur de recherche semantique pret')

Moteur de recherche semantique pret


In [8]:
# ── Build Response (identique au Modele 1) ──

def build_response(items, lang='en'):
    if not items:
        return {
            'fr': "Aucun resultat trouve. Essayez avec d'autres mots-cles.",
            'en': 'No results found. Try different keywords.',
            'ar': '\u0644\u0645 \u064a\u062a\u0645 \u0627\u0644\u0639\u062b\u0648\u0631 \u0639\u0644\u0649 \u0646\u062a\u0627\u0626\u062c. \u062d\u0627\u0648\u0644 \u0628\u0643\u0644\u0645\u0627\u062a \u0645\u0641\u062a\u0627\u062d\u064a\u0629 \u0623\u062e\u0631\u0649.',
        }.get(lang, 'No results found.')

    types = {}
    for item in items:
        types.setdefault(item['type'], []).append(item)

    parts = []
    total = len(items)
    labels = {
        'project': 'Projets' if lang == 'fr' else 'Projects',
        'stakeholder': 'Organisations' if lang == 'fr' else 'Organizations',
        'resource': 'Ressources' if lang == 'fr' else 'Resources',
    }

    if lang == 'fr':
        parts.append(f"{total} resultat{'s' if total>1 else ''} trouve{'s' if total>1 else ''}:")
    else:
        parts.append(f"{total} result{'s' if total!=1 else ''} found:")

    for t, entries in types.items():
        parts.append(f"\n**{labels.get(t, t.capitalize())}:**")
        for e in entries[:5]:
            ctx = [x for x in [e.get('country'), e.get('sector')] if x]
            suffix = f" ({', '.join(ctx)})" if ctx else ''
            parts.append(f"  - {e['title']}{suffix}")
        if len(entries) > 5:
            parts.append(f"  ... et {len(entries)-5} autre(s)" if lang=='fr' else f"  ... and {len(entries)-5} more")

    return '\n'.join(parts)

print('Build response OK')

Build response OK


In [9]:
# ── Orchestrateur principal ──

def semantic_search_chatbot(query, limit=10):
    """Point d'entree principal du chatbot semantique.

    Args:
        query: question de l'utilisateur
        limit: nombre max de resultats
    Returns:
        dict avec query, parsed, results, response, total_found
    """
    parsed = parse_query(query)
    entity = parsed['entity']
    country = parsed['country']
    sector = parsed['sector']
    tech = parsed['technology']
    lang = parsed['language']

    print(f'[Search] entity={entity} country={country} sector={sector} tech={tech}')

    # Embedding de la question
    query_emb = embedding_model.encode(query, normalize_embeddings=True)

    # Recherche semantique
    results, total = semantic_search(
        entities, query_emb,
        entity_type=entity,
        country=country,
        sector=sector,
        tech=tech,
        limit=limit
    )

    # Ajouter le score de similarite
    for r in results:
        r['score'] = float(np.dot(query_emb, r['embedding']))

    return {
        'query': query,
        'parsed': parsed,
        'results': results[:limit],
        'response': build_response(results[:limit], lang),
        'total_found': total,
    }

print('Orchestrateur semantique pret')

Orchestrateur semantique pret


---
## 4. Test du chatbot semantique

Les memes questions que le Modele 1, mais avec la recherche semantique.
On s'attend a de meilleurs resultats, notamment pour :
- "NLP projects in Morocco" → trouve "Adaptive Learning Platform" (qui utilise NLP) au Maroc
- "AI projects in UAE" → trouve les projets des Emirats
- "Machine learning resources" → trouve les ressources liees au ML

In [10]:
# ── Execution des requetes de test ──

example_queries = [
    'Quels projets IA en sante en Tunisie ?',
    'NLP projects in Morocco',
    'Show me AI startups in Egypt',
    'Machine learning resources',
    'AI healthcare projects',
    'Arabic speech recognition',
    'AI projects in UAE',
]

print('TEST DU SEMANTIC SEARCH CHATBOT (Modele 2)')
print('='*70)
for q in example_queries:
    print(f"\n{'='*70}")
    print(f'QUESTION: {q}')
    print('='*70)
    start = time.time()
    result = semantic_search_chatbot(q, limit=5)
    elapsed = int((time.time() - start) * 1000)
    print(f'\nREPONSE ({elapsed}ms):')
    print(result['response'])
    if result['results']:
        n = len(result['results'])
        print(f"\nDetails: {n} affiche(s) / {result['total_found']} trouve(s)")
        for r in result['results']:
            print(f"  [{r['type']:12}] score={r['score']:.4f} | {r['title']} ({r['country']})")

TEST DU SEMANTIC SEARCH CHATBOT (Modele 2)

QUESTION: Quels projets IA en sante en Tunisie ?
[Search] entity=project country=Tunisia sector=Health tech=None

REPONSE (18ms):
Aucun resultat trouve. Essayez avec d'autres mots-cles.

QUESTION: NLP projects in Morocco
[Search] entity=project country=Morocco sector=None tech=NLP

REPONSE (19ms):
No results found. Try different keywords.

QUESTION: Show me AI startups in Egypt
[Search] entity=stakeholder country=Egypt sector=None tech=None

REPONSE (8ms):
2 results found:

**Organizations:**
  - Cairo University AI Lab (Egypt, University)
  - Alexandria University AI Center (Egypt, University)

Details: 2 affiche(s) / 2 trouve(s)
  [stakeholder ] score=0.6324 | Cairo University AI Lab (Egypt)
  [stakeholder ] score=0.5962 | Alexandria University AI Center (Egypt)

QUESTION: Machine learning resources
[Search] entity=resource country=None sector=Education tech=Machine Learning

REPONSE (9ms):
No results found. Try different keywords.

QUESTIO

---
## 5. Evaluation du Modele 2

Test set elargi (8 queries) avec des vrais cas semantiques.
Les scores 100% sont impossibles ici car :
- Plusieurs queries n'ont **aucun filtre** (entity=None, country=None)
  → les modeles classent parmi les **26 entites**
- k=5 est petit → Precision et Recall penalises si le modele ne ranke pas bien
- Metriques incluses : Precision@k, Recall@k, F1@k, **MRR** (Mean Reciprocal Rank)

In [11]:
# ── Test set realiste (8 queries semantiques) ──

test_set = [
    # Q1: Pure semantic (aucun filtre) → cherche sur 26 entites
    ("Innovative artificial intelligence solutions in Arab region", {
        ('project', 1), ('project', 3), ('project', 5), ('project', 9),
        ('stakeholder', 1), ('stakeholder', 2), ('stakeholder', 6),
        ('stakeholder', 9), ('stakeholder', 10),
        ('resource', 1), ('resource', 4),
    }),
    # Q2: entity=project, country=UAE → keyword ne trouve pas (pas de 'ai' dans titres UAE)
    ("AI projects in UAE", {
        ('project', 3), ('project', 5),
    }),
    # Q3: Pure semantic → arabe
    ("Arabic language technologies", {
        ('project', 6), ('project', 9), ('resource', 3),
    }),
    # Q4: Pure semantic → finance (aucun mot-filtre, tout le corpus)
    ("Banking and financial technology innovations", {
        ('project', 5), ('project', 12),
    }),
    # Q5: Pure semantic → ethique & gouvernance (aucun filtre)
    ("AI ethics and governance recommendations", {
        ('resource', 2), ('resource', 5), ('stakeholder', 12),
    }),
    # Q6: entity=stakeholder → labs de recherche (filtre partiel)
    ("AI research centers and laboratories", {
        ('stakeholder', 1), ('stakeholder', 6), ('stakeholder', 9),
    }),
    # Q7: Pure semantic → environnement + eau
    ("Environmental monitoring and waste management systems", {
        ('project', 8), ('project', 11),
    }),
    # Q8: Pure semantic → agriculture + irrigation
    ("Smart farming and crop irrigation technology", {
        ('project', 2),
    }),
]


def reciprocal_rank(retrieved_list, expected):
    """MRR : 1/rank du premier relevant trouve, 0 si aucun."""
    for rank, item in enumerate(retrieved_list, start=1):
        key = (item['type'], item['id'])
        if key in expected:
            return 1.0 / rank
    return 0.0


def evaluate(search_fn, test_set, k=5):
    results = []
    for query, expected in test_set:
        if not expected:
            continue
        result = search_fn(query, limit=k)
        retrieved_list = result['results']
        retrieved = set((r['type'], r['id']) for r in retrieved_list)

        tp = len(retrieved & expected)
        fp = len(retrieved - expected)
        fn_c = len(expected - retrieved)

        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn_c) if (tp + fn_c) > 0 else 0.0
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        mrr = reciprocal_rank(retrieved_list, expected)

        results.append({
            'query': query[:55],
            'precision': round(p, 3),
            'recall': round(r, 3),
            'f1': round(f1, 3),
            'mrr': round(mrr, 3),
            'retrieved': len(retrieved),
            'expected': len(expected),
            'tp': tp, 'fp': fp, 'fn': fn_c,
        })
    return results


results = evaluate(semantic_search_chatbot, test_set, k=5)
if results:
    print(f"{'Query':<55} {'P@5':<8} {'R@5':<8} {'F1@5':<8} {'MRR':<8} {'Ret':<6} {'Exp':<6}")
    print('-'*93)
    for r in results:
        print(f"{r['query']:<55} {r['precision']:<8.3f} {r['recall']:<8.3f} {r['f1']:<8.3f} {r['mrr']:<8.3f} {r['retrieved']:<6} {r['expected']:<6}")
    avg_p = sum(r['precision'] for r in results) / len(results)
    avg_r = sum(r['recall'] for r in results) / len(results)
    avg_f = sum(r['f1'] for r in results) / len(results)
    avg_mrr = sum(r['mrr'] for r in results) / len(results)
    print(f"\n{'MOYENNE':<55} {avg_p:<8.3f} {avg_r:<8.3f} {avg_f:<8.3f} {avg_mrr:<8.3f}")
    print(f"\n→ Les scores NE sont PAS 100% car le test set est realiste.")
else:
    print('Test set vide ou sans expected IDs.')

[Search] entity=None country=None sector=None tech=None
[Search] entity=project country=UAE sector=None tech=None
[Search] entity=None country=None sector=None tech=None
[Search] entity=None country=None sector=Finance tech=None
[Search] entity=None country=None sector=None tech=None
[Search] entity=stakeholder country=None sector=None tech=None
[Search] entity=None country=None sector=Environment tech=None
[Search] entity=None country=None sector=Agriculture tech=None
Query                                                   P@5      R@5      F1@5     MRR      Ret    Exp   
---------------------------------------------------------------------------------------------
Innovative artificial intelligence solutions in Arab re 0.800    0.364    0.500    1.000    5      11    
AI projects in UAE                                      1.000    1.000    1.000    1.000    2      2     
Arabic language technologies                            0.400    0.667    0.500    1.000    5      3     
Banking 

---
## 5b. Evaluation approfondie

Metriques : Precision, Recall, F1, **MRR**, Accuracy, Semantic Similarity, Response Time.
(ROUGE-L et BLEU supprimes : inadaptes a un modele de recherche, reserve a la generation de texte)
k=5 pour rendre les scores discriminants.

In [12]:
# ── Evaluation approfondie ──

def eval_advanced(search_fn, test_set, k=5):
    results = []
    for query, expected_ids in test_set:
        if not expected_ids:
            continue
        start = time.time()
        result = search_fn(query, limit=k)
        elapsed = int((time.time() - start) * 1000)
        retrieved_list = result['results']
        retrieved = set((r['type'], r['id']) for r in retrieved_list)
        response_text = result['response']
        tp = len(retrieved & expected_ids)
        fp = len(retrieved - expected_ids)
        fn_c = len(expected_ids - retrieved)
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r_c = tp / (tp + fn_c) if (tp + fn_c) > 0 else 0.0
        f1 = 2 * p * r_c / (p + r_c) if (p + r_c) > 0 else 0.0
        acc = tp / (tp + fp + fn_c) if (tp + fp + fn_c) > 0 else 0.0
        mrr = 0.0
        for rank, item in enumerate(retrieved_list, start=1):
            if (item['type'], item['id']) in expected_ids:
                mrr = 1.0 / rank
                break
        ss = None
        if response_text:
            emb_q = embedding_model.encode(query, normalize_embeddings=True)
            emb_r = embedding_model.encode(response_text, normalize_embeddings=True)
            ss = round(float(np.dot(emb_q, emb_r)), 4)
        results.append({
            'query': query[:45],
            'accuracy': round(acc, 3),
            'precision': round(p, 3),
            'recall': round(r_c, 3),
            'f1': round(f1, 3),
            'mrr': round(mrr, 3),
            'sem_sim': ss,
            'time_ms': elapsed,
        })
    return results


print('='*115)
print('EVALUATION APPROFONDIE DU MODELE 2 (SEMANTIC SEARCH)')
print('='*115)

adv_results = eval_advanced(semantic_search_chatbot, test_set, k=5)

if not adv_results:
    print('\nTest set vide.')
else:
    header = f"{'Query':<45} {'Acc':<8} {'P@5':<8} {'R@5':<8} {'F1@5':<8} {'MRR':<8} {'SemSim':<8} {'Time':<8}"
    print(header)
    print('-' * len(header))
    totals = {'acc': [], 'p': [], 'r': [], 'f1': [], 'mrr': [], 'ss': [], 't': []}
    for r in adv_results:
        ss_str = f"{r['sem_sim']:.3f}" if r['sem_sim'] is not None else 'N/A'
        print(f"{r['query']:<45} {r['accuracy']:<8.3f} {r['precision']:<8.3f} {r['recall']:<8.3f} {r['f1']:<8.3f} {r['mrr']:<8.3f} {ss_str:<8} {r['time_ms']:<8}")
        totals['acc'].append(r['accuracy'])
        totals['p'].append(r['precision'])
        totals['r'].append(r['recall'])
        totals['f1'].append(r['f1'])
        totals['mrr'].append(r['mrr'])
        if r['sem_sim'] is not None: totals['ss'].append(r['sem_sim'])
        totals['t'].append(r['time_ms'])
    avg = lambda lst: sum(lst)/len(lst) if lst else 0.0
    ss_avg = f"{avg(totals['ss']):.3f}" if totals['ss'] else 'N/A'
    print('-' * len(header))
    print(f"{'MOYENNE':<45} {avg(totals['acc']):<8.3f} {avg(totals['p']):<8.3f} {avg(totals['r']):<8.3f} {avg(totals['f1']):<8.3f} {avg(totals['mrr']):<8.3f} {ss_avg:<8} {avg(totals['t']):<8.0f}")
    print('\n→ Scores realistes : Precision/Recall/F1 ne sont PAS 100%')

EVALUATION APPROFONDIE DU MODELE 2 (SEMANTIC SEARCH)
[Search] entity=None country=None sector=None tech=None
[Search] entity=project country=UAE sector=None tech=None
[Search] entity=None country=None sector=None tech=None
[Search] entity=None country=None sector=Finance tech=None
[Search] entity=None country=None sector=None tech=None
[Search] entity=stakeholder country=None sector=None tech=None
[Search] entity=None country=None sector=Environment tech=None
[Search] entity=None country=None sector=Agriculture tech=None
Query                                         Acc      P@5      R@5      F1@5     MRR      SemSim   Time    
------------------------------------------------------------------------------------------------------------
Innovative artificial intelligence solutions  0.333    0.800    0.364    0.500    1.000    0.668    21      
AI projects in UAE                            1.000    1.000    1.000    1.000    1.000    0.547    0       
Arabic language technologies         

---
## 5c. Metriques completes du Modele 2 (Precision, Recall, F1, MRR, NDCG, Temps)

Section dediee au calcul de **toutes** les metriques pour le Modele 2 :
- Precision@k, Recall@k, F1@k
- MRR (Mean Reciprocal Rank)
- **NDCG@k** (Normalized Discounted Cumulative Gain)
- Temps de reponse moyen

In [13]:
# --- Metriques completes ---

import math

def ndcg_at_k(retrieved_list, expected, k=5):
    dcg = 0.0
    for i, item in enumerate(retrieved_list[:k]):
        rel = 1 if (item['type'], item['id']) in expected else 0
        dcg += (2**rel - 1) / math.log2(i + 2)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(min(k, len(expected))))
    return dcg / idcg if idcg > 0 else 0.0

def evaluate_all_metrics(search_fn, test_set, k=5):
    results = []
    for query, expected in test_set:
        if not expected:
            continue
        start = time.time()
        result = search_fn(query, limit=k)
        elapsed = int((time.time() - start) * 1000)
        retrieved_list = result['results']
        retrieved = set((r['type'], r['id']) for r in retrieved_list)

        tp = len(retrieved & expected)
        fp = len(retrieved - expected)
        fn_c = len(expected - retrieved)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn_c) if (tp + fn_c) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

        mrr = 0.0
        for rank, item in enumerate(retrieved_list, start=1):
            if (item['type'], item['id']) in expected:
                mrr = 1.0 / rank
                break

        ndcg = ndcg_at_k(retrieved_list, expected, k)

        results.append({
            'query': query[:55],
            'precision': round(precision, 3),
            'recall': round(recall, 3),
            'f1': round(f1, 3),
            'mrr': round(mrr, 3),
            'ndcg': round(ndcg, 3),
            'time_ms': elapsed,
            'retrieved': len(retrieved),
            'expected': len(expected),
        })
    return results

print('='*115)
print('METRIQUES COMPLETES DU MODELE 2 (SEMANTIC SEARCH)')
print('='*115)
print()

all_results = evaluate_all_metrics(semantic_search_chatbot, test_set, k=5)

if all_results:
    header = f"{'Query':<55} {'P@5':<8} {'R@5':<8} {'F1@5':<8} {'MRR':<8} {'NDCG@5':<8} {'Time(ms)':<8} {'Ret':<6} {'Exp':<6}"
    print(header)
    print('-' * len(header))
    for r in all_results:
        print(f"{r['query']:<55} {r['precision']:<8.3f} {r['recall']:<8.3f} {r['f1']:<8.3f} {r['mrr']:<8.3f} {r['ndcg']:<8.3f} {r['time_ms']:<8} {r['retrieved']:<6} {r['expected']:<6}")

    avg_p = sum(r['precision'] for r in all_results) / len(all_results)
    avg_r = sum(r['recall'] for r in all_results) / len(all_results)
    avg_f = sum(r['f1'] for r in all_results) / len(all_results)
    avg_mrr = sum(r['mrr'] for r in all_results) / len(all_results)
    avg_ndcg = sum(r['ndcg'] for r in all_results) / len(all_results)
    avg_t = sum(r['time_ms'] for r in all_results) / len(all_results)

    print('-' * len(header))
    print(f"{'MOYENNE':<55} {avg_p:<8.3f} {avg_r:<8.3f} {avg_f:<8.3f} {avg_mrr:<8.3f} {avg_ndcg:<8.3f} {avg_t:<8.0f}")
else:
    print('Aucun resultat a evaluer.')

print(f"\n→ NDCG penalise le rang : 1.0 si tous les resultats pertinents sont en haut, 0.0 sinon.")
print(f"→ k=5 : seuls les 5 premiers resultats sont consideres.")

METRIQUES COMPLETES DU MODELE 2 (SEMANTIC SEARCH)

[Search] entity=None country=None sector=None tech=None
[Search] entity=project country=UAE sector=None tech=None
[Search] entity=None country=None sector=None tech=None
[Search] entity=None country=None sector=Finance tech=None
[Search] entity=None country=None sector=None tech=None
[Search] entity=stakeholder country=None sector=None tech=None
[Search] entity=None country=None sector=Environment tech=None
[Search] entity=None country=None sector=Agriculture tech=None
Query                                                   P@5      R@5      F1@5     MRR      NDCG@5   Time(ms) Ret    Exp   
---------------------------------------------------------------------------------------------------------------------------
Innovative artificial intelligence solutions in Arab re 0.800    0.364    0.500    1.000    0.854    15       5      11    
AI projects in UAE                                      1.000    1.000    1.000    1.000    1.000    50

---
## 6. Comparaison Modele 1 vs Modele 2

Execute les deux modeles sur le meme test set et compare les performances.

In [14]:
# ── Import du Modele 1 pour comparaison ──

# On importe les fonctions du notebook keyword_chatbot
# On redefinit ici les fonctions essentielles du Modele 1 pour pouvoir comparer

# Reuse de la session DB existante pour le Modele 1
def keyword_search_model1(session, query, limit=10):
    """Modele 1 : Keyword search (ILKE).
    Version simplifiee qui utilise directement les fonctions ci-dessous.
    """
    parsed = parse_query(query)
    kw = parsed['keywords']
    country = parsed['country']
    entity = parsed['entity']
    sector = parsed['sector']
    tech = parsed['technology']
    lang = parsed['language']

    # Retirer des keywords les mots deja consommes comme filtres
    filter_words_lower = set()
    if country:
        for alias, c in COUNTRY_MAP.items():
            if c == country:
                filter_words_lower.update(alias.split())
    if sector and sector.lower():
        sec_lower = sector.lower()
        for sec, keywords_set in SECTOR_MAP.items():
            if sec.lower() == sec_lower or sec.capitalize() == sector:
                for alias in keywords_set:
                    filter_words_lower.update(alias.split())
    if tech:
        for alias, t in TECH_MAP.items():
            if t == tech:
                filter_words_lower.update(alias.split())
    kw = [w for w in kw if w not in filter_words_lower]
    parsed['keywords'] = kw

    all_r = []
    if entity is None or entity == 'project':
        all_r.extend(search_projects_ilike(session, kw, country, sector, tech, limit))
    if entity is None or entity == 'stakeholder':
        all_r.extend(search_stakeholders_ilike(session, kw, country, limit))
    if entity is None or entity == 'resource':
        all_r.extend(search_resources_ilike(session, kw, limit))

    all_r.sort(key=lambda x: x['score'], reverse=True)
    seen = set()
    unique = []
    for r in all_r:
        key = (r['type'], r['id'])
        if key not in seen:
            seen.add(key)
            unique.append(r)

    return {
        'query': query,
        'parsed': parsed,
        'results': unique[:limit],
        'response': build_response(unique[:limit], lang),
        'total_found': len(unique),
    }


def build_ilike_conditions(params, alias, columns, keywords, prefix='w'):
    if not keywords:
        return '1=1', params
    clauses = []
    for i, kw in enumerate(keywords):
        for j, col in enumerate(columns):
            pname = f'{prefix}_{i}_{j}'
            clauses.append(f"LOWER({alias}.{col}) LIKE :{pname}")
            params[pname] = f'%{kw}%'
    return '(' + ' OR '.join(clauses) + ')', params


def count_matches(keywords, *text_fields):
    if not keywords:
        return 1
    return sum(1 for kw in keywords
               if any(kw in (t or '').lower() for t in text_fields))


def search_projects_ilike(s, keywords, country=None, sector=None, tech=None, limit=10):
    if not keywords and not country and not sector and not tech:
        return []
    params = {}
    where = []
    if keywords:
        cond, params = build_ilike_conditions(params, 'p',
            ['title','description','sector','technology'], keywords)
        where.append(cond)
    if country:
        where.append('LOWER(c.country) LIKE :country')
        params['country'] = f'%{country.lower()}%'
    if sector:
        where.append('(LOWER(p.sector) = :sector OR LOWER(p.sector) LIKE :sector_like)')
        params['sector'] = sector.lower()
        params['sector_like'] = f'%{sector.lower()}%'
    if tech:
        where.append('(LOWER(p.technology) = :tech OR LOWER(p.technology) LIKE :tech_like)')
        params['tech'] = tech.lower()
        params['tech_like'] = f'%{tech.lower()}%'
    where.append("p.status NOT IN ('pending','rejected')")
    sql = f'''
        SELECT p.id,p.title,p.description,p.sector,p.technology,
               p.year_of_implementation,p.organization,
               c.country AS country
        FROM projects p
        LEFT JOIN countries c ON c.id = p.country_id
        WHERE {' AND '.join(where)}
        ORDER BY p.updated_at DESC
        LIMIT :lim
    '''
    params['lim'] = limit
    rows = s.execute(text(sql), params).fetchall()
    n = max(len(keywords), 1)
    return [{
        'id': r.id, 'title': r.title, 'type': 'project',
        'country': r.country or '', 'sector': r.sector or '',
        'technology': r.technology or '',
        'description': (r.description or '')[:200],
        'score': min(1.0, count_matches(keywords, r.title, r.description, r.sector, r.technology) / n
                     + (0.2 if count_matches(keywords, r.title, r.description, r.sector, r.technology) == n and keywords else 0)),
    } for r in rows]


def search_stakeholders_ilike(s, keywords, country=None, limit=10):
    if not keywords and not country:
        return []
    params = {}
    where = []
    if keywords:
        cond, params = build_ilike_conditions(params, 's',
            ['name','description','type','country'], keywords, 'sw')
        where.append(cond)
    if country:
        where.append('LOWER(s.country) LIKE :country')
        params['country'] = f'%{country.lower()}%'
    sql = f'''
        SELECT s.id,s.name AS title,s.description,s.type,s.country,s.category
        FROM stakeholders s
        WHERE {' AND '.join(where)}
        ORDER BY s.updated_at DESC
        LIMIT :lim
    '''
    params['lim'] = limit
    rows = s.execute(text(sql), params).fetchall()
    n = max(len(keywords), 1)
    return [{
        'id': r.id, 'title': r.title, 'type': 'stakeholder',
        'country': r.country or '', 'sector': r.type or '',
        'description': (r.description or '')[:200],
        'score': count_matches(keywords, r.title, r.description, r.type, r.country) / n if keywords else 0.5,
    } for r in rows]


def search_resources_ilike(s, keywords, limit=10):
    if not keywords:
        return []
    params = {}
    cond, params = build_ilike_conditions(params, 'r',
        ['title','description','category','type'], keywords, 'rw')
    sql = f'''
        SELECT r.id,r.title,r.description,r.type,r.category,r.language
        FROM resources r
        WHERE {cond}
        ORDER BY r.updated_at DESC
        LIMIT :lim
    '''
    params['lim'] = limit
    rows = s.execute(text(sql), params).fetchall()
    n = max(len(keywords), 1)
    return [{
        'id': r.id, 'title': r.title, 'type': 'resource',
        'country': '', 'sector': r.category or '',
        'description': (r.description or '')[:200],
        'score': count_matches(keywords, r.title, r.description, r.category, r.type) / n,
    } for r in rows]


# Wrapper pour utiliser la meme signature que semantic_search_chatbot
def keyword_search_chatbot(query, limit=10):
    return keyword_search_model1(db, query, limit)


print('Modele 1 (Keyword Search) importe pour comparaison')
print('Modele 2 (Semantic Search) deja charge')

Modele 1 (Keyword Search) importe pour comparaison
Modele 2 (Semantic Search) deja charge


In [15]:
# ── Execution de la comparaison ──

def compare_models(test_set, k=5):
    models = {
        '1. Keyword Search': keyword_search_chatbot,
        '2. Semantic Search': semantic_search_chatbot,
    }

    print(f"\n{'='*102}")
    print("COMPARAISON MODELE 1 (KEYWORD) vs MODELE 2 (SEMANTIQUE)")
    print('='*102)
    print(f"{'Modele':<22} {'P@5':<10} {'R@5':<10} {'F1@5':<10} {'MRR':<10} {'Temps(ms)':<12} {'Ret':<6} {'Exp':<6}")
    print('-'*102)

    for name, fn in models.items():
        precisions, recalls, f1s, mrrs, times = [], [], [], [], []
        total_retrieved = 0
        total_expected = 0
        for query, expected in test_set:
            if not expected:
                continue
            start = time.time()
            result = fn(query, limit=k)
            elapsed = int((time.time() - start) * 1000)
            retrieved_list = result['results']
            retrieved = set((r['type'], r['id']) for r in retrieved_list)
            tp = len(retrieved & expected)
            fp = len(retrieved - expected)
            fn_c = len(expected - retrieved)
            p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            r = tp / (tp + fn_c) if (tp + fn_c) > 0 else 0.0
            f = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
            mrr_val = 0.0
            for rank, item in enumerate(retrieved_list, start=1):
                if (item['type'], item['id']) in expected:
                    mrr_val = 1.0 / rank
                    break
            precisions.append(p)
            recalls.append(r)
            f1s.append(f)
            mrrs.append(mrr_val)
            times.append(elapsed)
            total_retrieved += len(retrieved)
            total_expected += len(expected)
        avg_p = sum(precisions) / len(precisions) if precisions else 0
        avg_r = sum(recalls) / len(recalls) if recalls else 0
        avg_f = sum(f1s) / len(f1s) if f1s else 0
        avg_mrr = sum(mrrs) / len(mrrs) if mrrs else 0
        avg_t = sum(times) / len(times) if times else 0
        print(f"{name:<22} {avg_p:<10.3f} {avg_r:<10.3f} {avg_f:<10.3f} {avg_mrr:<10.3f} {avg_t:<12.0f} {total_retrieved:<6} {total_expected:<6}")

    print('\n' + '='*102)
    print("CONCLUSION :")
    print("- Modele 1 (Keyword) : cherche des mots exacts → rate les documents qui utilisent des synonymes")
    print("- Modele 2 (Semantic) : comprend le sens → meilleur rappel, peut trouver sans mots communs")
    print("- MRR penalise le rang : un resultat pertinent en 1ere position donne MRR=1.0, en 3e donne MRR=0.33")
    print('='*102)


compare_models(test_set, k=5)


COMPARAISON MODELE 1 (KEYWORD) vs MODELE 2 (SEMANTIQUE)
Modele                 P@5        R@5        F1@5       MRR        Temps(ms)    Ret    Exp   
------------------------------------------------------------------------------------------------------
1. Keyword Search      0.525      0.460      0.465      0.667      2            19     27    
[Search] entity=None country=None sector=None tech=None
[Search] entity=project country=UAE sector=None tech=None
[Search] entity=None country=None sector=None tech=None
[Search] entity=None country=None sector=Finance tech=None
[Search] entity=None country=None sector=None tech=None
[Search] entity=stakeholder country=None sector=None tech=None
[Search] entity=None country=None sector=Environment tech=None
[Search] entity=None country=None sector=Agriculture tech=None
2. Semantic Search     0.775      0.775      0.740      1.000      7            26     27    

CONCLUSION :
- Modele 1 (Keyword) : cherche des mots exacts → rate les documents qu

---
## 6b. Comparaison Approfondie

Execute les deux modeles cote-a-cote sur chaque question du test set
et montre les resultats detailles.

In [16]:
# ── Comparaison question par question ──

comparison_queries = [
    'Innovative artificial intelligence solutions in Arab region',
    'AI projects in UAE',
    'Arabic language technologies',
    'Banking and financial technology innovations',
    'AI ethics and governance recommendations',
    'Environmental monitoring and waste management systems',
    'Smart farming and crop irrigation technology',
    'AI research centers and laboratories',
]

print('='*120)
print("COMPARAISON DETAILLEE : MODELE 1 (KEYWORD) vs MODELE 2 (SEMANTIQUE)")
print('='*120)

for q in comparison_queries:
    print(f"\n{'─'*120}")
    print(f"QUESTION : {q}")
    print('─'*120)

    # Modele 1
    start = time.time()
    r1 = keyword_search_chatbot(q, limit=5)
    t1 = int((time.time() - start) * 1000)

    # Modele 2
    start = time.time()
    r2 = semantic_search_chatbot(q, limit=5)
    t2 = int((time.time() - start) * 1000)

    # Affichage
    p1 = r1['parsed']
    p2 = r2['parsed']
    print(f"  Parsed: country={p1['country']} entity={p1['entity']} sector={p1['sector']} tech={p1['technology']}")
    print(f"")
    print(f"  {'MODELE 1 (Keyword)':<40} {'MODELE 2 (Semantic)':<40}")
    print(f"  {'─'*39} {'─'*39}")
    print(f"  {'Temps: '+str(t1)+'ms':<40} {'Temps: '+str(t2)+'ms':<40}")
    print(f"  {'Trouves: '+str(r1['total_found']):<40} {'Trouves: '+str(r2['total_found']):<40}")
    print(f"")

    # Resultats
    for i in range(max(len(r1['results']), len(r2['results']))):
        left = ''
        right = ''
        if i < len(r1['results']):
            item = r1['results'][i]
            left = f"{item['score']:.3f} | {item['title']} ({item['country']})"
        if i < len(r2['results']):
            item = r2['results'][i]
            right = f"{item['score']:.4f} | {item['title']} ({item['country']})"
        print(f"  {left:<40} {right:<40}")
    print()

COMPARAISON DETAILLEE : MODELE 1 (KEYWORD) vs MODELE 2 (SEMANTIQUE)

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
QUESTION : Innovative artificial intelligence solutions in Arab region
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
[Search] entity=None country=None sector=None tech=None
  Parsed: country=None entity=None sector=None tech=None

  MODELE 1 (Keyword)                       MODELE 2 (Semantic)                     
  ─────────────────────────────────────── ───────────────────────────────────────
  Temps: 3ms                               Temps: 12ms                             
  Trouves: 5                               Trouves: 26                             

  0.167 | Arabic Speech Recognition (Qatar) 0.6828 | Arab Common AI Strategy 2023 ()
  0.167 | Saudi Data Authority (Saudi Arabia) 0.6607 | Dubai AI Center (UAE)        

---
## 7. Chat interactif

Posez vos questions en continu. Tapez `quit`, `exit` ou `q` pour quitter.

In [17]:
print("=== SEMANTIC SEARCH CHATBOT (Modele 2) ===")
print("Tapez 'quit', 'exit' ou 'q' pour quitter.\n")

while True:
    q = input("\nVotre question: ").strip()
    if q.lower() in ('quit', 'exit', 'q'):
        print("Au revoir!")
        break
    if not q:
        continue

    start = time.time()
    result = semantic_search_chatbot(q, limit=5)
    elapsed = int((time.time() - start) * 1000)

    print(f"\nTemps: {elapsed}ms")
    print(f"Reponse: {result['response']}")
    if result['results']:
        p = result['parsed']
        print(f"Parsed: pays={p['country']} type={p['entity']} secteur={p['sector']} tech={p['technology']}")
        print(f"Resultats: {len(result['results'])} affiche(s) / {result['total_found']} trouve(s)")
        for r in result['results']:
            print(f"  [{r['type']:12}] score={r['score']:.4f} | {r['title']} ({r['country']})")
    print('-' * 60)

=== SEMANTIC SEARCH CHATBOT (Modele 2) ===
Tapez 'quit', 'exit' ou 'q' pour quitter.

[Search] entity=None country=Tunisia sector=None tech=NLP

Temps: 4ms
Reponse: No results found. Try different keywords.
------------------------------------------------------------
[Search] entity=project country=Tunisia sector=None tech=NLP

Temps: 12ms
Reponse: No results found. Try different keywords.
------------------------------------------------------------
[Search] entity=project country=UAE sector=None tech=NLP

Temps: 0ms
Reponse: 1 result found:

**Projects:**
  - Adaptive Learning Platform (UAE, Education)
Parsed: pays=UAE type=project secteur=None tech=NLP
Resultats: 1 affiche(s) / 1 trouve(s)
  [project     ] score=0.5831 | Adaptive Learning Platform (UAE)
------------------------------------------------------------
Au revoir!


---
## 8. Fermeture de la connexion

In [ ]:
db.close()
print('Connexion DB fermee.')